In [2]:
import pandas as pd

df = pd.read_csv("BankChurners.csv")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully!
Rows: 10127
Columns: 23


In [3]:
# Remove irrelevant columns
columns_to_remove = [
    "CLIENTNUM",
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1",
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2"
]

df = df.drop(columns=columns_to_remove)

# Create the same engineered features used in Part B
df["Avg_Transaction_Amount"] = (
    df["Total_Trans_Amt"] / df["Total_Trans_Ct"]
)

df["High_Inactivity"] = (
    df["Months_Inactive_12_mon"] >= 3
).astype(int)

df["Transaction_Activity"] = (
    df["Total_Trans_Ct"] * df["Total_Trans_Amt"]
)

print("Data preparation completed.")
print("Remaining columns:", df.shape[1])

Data preparation completed.
Remaining columns: 23


In [4]:
print(df.columns.tolist())

['Attrition_Flag', 'Customer_Age', 'Gender', 'Dependent_count', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio', 'Avg_Transaction_Amount', 'High_Inactivity', 'Transaction_Activity']


In [5]:
# Separate target and predictor variables

X = df.drop(columns=["Attrition_Flag"])
y = df["Attrition_Flag"]

# Convert target to numerical values
y = y.map({
    "Existing Customer": 0,
    "Attrited Customer": 1
})

print("Features:", X.shape[1])
print("Target values:")
print(y.value_counts())

Features: 22
Target values:
Attrition_Flag
0    8500
1    1627
Name: count, dtype: int64


In [6]:
# Identify categorical variables
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

# One-hot encode categorical variables
X = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True
)

print("\nFeatures after one-hot encoding:", X.shape[1])

Categorical columns:
['Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category']

Features after one-hot encoding: 35


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 8101
Testing samples: 2026


In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Scale the features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Logistic Regression
model1 = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model1.fit(X_train_scaled, y_train)

print("Model 1 trained successfully.")

Model 1 trained successfully.


In [9]:
# Generate predictions
y_pred1 = model1.predict(X_test_scaled)
y_prob1 = model1.predict_proba(X_test_scaled)[:, 1]

print("Model 1 predictions generated successfully.")

Model 1 predictions generated successfully.


In [10]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

accuracy1 = accuracy_score(y_test, y_pred1)
precision1 = precision_score(y_test, y_pred1)
recall1 = recall_score(y_test, y_pred1)
f1_1 = f1_score(y_test, y_pred1)
roc_auc1 = roc_auc_score(y_test, y_prob1)

print("MODEL 1 - LOGISTIC REGRESSION")
print("--------------------------------")
print("Accuracy :", round(accuracy1, 4))
print("Precision:", round(precision1, 4))
print("Recall   :", round(recall1, 4))
print("F1-score :", round(f1_1, 4))
print("ROC-AUC  :", round(roc_auc1, 4))

MODEL 1 - LOGISTIC REGRESSION
--------------------------------
Accuracy : 0.8657
Precision: 0.5538
Recall   : 0.84
F1-score : 0.6675
ROC-AUC  : 0.9421


In [11]:
from xgboost import XGBClassifier

# Calculate class imbalance from the training data
negative_class = (y_train == 0).sum()
positive_class = (y_train == 1).sum()

scale_pos_weight = negative_class / positive_class

print("Existing Customers:", negative_class)
print("Attrited Customers:", positive_class)
print("Scale pos weight:", round(scale_pos_weight, 2))

Existing Customers: 6799
Attrited Customers: 1302
Scale pos weight: 5.22


In [12]:
model2 = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=5.22,
    random_state=42,
    eval_metric="logloss"
)

model2.fit(X_train, y_train)

print("Model 2 trained successfully.")

Model 2 trained successfully.


In [13]:
# Generate Model 2 predictions
y_pred2 = model2.predict(X_test)
y_prob2 = model2.predict_proba(X_test)[:, 1]

print("Model 2 predictions generated successfully.")

Model 2 predictions generated successfully.


In [14]:
accuracy2 = accuracy_score(y_test, y_pred2)
precision2 = precision_score(y_test, y_pred2)
recall2 = recall_score(y_test, y_pred2)
f1_2 = f1_score(y_test, y_pred2)
roc_auc2 = roc_auc_score(y_test, y_prob2)

print("MODEL 2 - XGBOOST")
print("-----------------")
print("Accuracy :", round(accuracy2, 4))
print("Precision:", round(precision2, 4))
print("Recall   :", round(recall2, 4))
print("F1-score :", round(f1_2, 4))
print("ROC-AUC  :", round(roc_auc2, 4))

MODEL 2 - XGBOOST
-----------------
Accuracy : 0.963
Precision: 0.8592
Recall   : 0.92
F1-score : 0.8886
ROC-AUC  : 0.9924


In [15]:
results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],
    "Logistic Regression": [
        accuracy1,
        precision1,
        recall1,
        f1_1,
        roc_auc1
    ],
    "XGBoost": [
        accuracy2,
        precision2,
        recall2,
        f1_2,
        roc_auc2
    ]
})

results

,Metric,Logistic Regression,XGBoost
0,Accuracy,0.865745,0.962981
1,Precision,0.553753,0.859195
2,Recall,0.840000,0.920000
3,F1-score,0.667482,0.888559
4,ROC-AUC,0.942094,0.992386
